In [2]:
import numpy as np
from embedder import Embedder

2026-06-25 19:46:09.208810989 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [3]:
embedder = Embedder()

Question 1

In [4]:
question="How does approximate nearest neighbor search work?"

vec=embedder.encode(question)

In [5]:
len(vec)

384

In [6]:
vec[0]

np.float64(-0.02058203437252893)

Question 2

In [7]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [27]:
doc = next(
    d for d in documents
    if d["filename"] ==
    "02-vector-search/lessons/07-sqlitesearch-vector.md"
)
doc

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [30]:
doc_vec=embedder.encode(doc["content"])


In [31]:
doc_vec.dot(vec)

np.float64(0.36107026789538205)

Question 3

In [32]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [33]:
len(chunks)

295

In [34]:
vecs = embedder.encode_batch(
    [c["content"] for c in chunks]
)

In [35]:
X = np.vstack(vecs)

In [38]:
scores = X.dot(vec)

In [40]:
idx=scores.argmax()

In [41]:
chunks[idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

Question 4

In [46]:
from minsearch import VectorSearch

vecindex=VectorSearch(keyword_fields=["course"])
vecindex.fit(X, chunks)

In [47]:
question = "What metric do we use to evaluate a search engine?"
que_vector=embedder.encode(question)

results=vecindex.search(que_vector)

In [49]:
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

Question 5

In [50]:
##Text

from minsearch import Index

index=Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [51]:
question2="How do I store vectors in PostgreSQL?"

In [65]:
t_results=index.search(
    query=question2,
    num_results=5
)

for i in t_results:
    print(i["filename"])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [54]:
##Vector

vec2=embedder.encode(question2)

In [67]:
v_results=vecindex.search(vec2, num_results=5)

for i in v_results:
    print(i["filename"])


02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


Question 6

In [71]:
final_question="How do I give the model access to tools?"

In [72]:
##Text
text_results=index.search(final_question,num_results=5)

for i in text_results:
    print(i["filename"])

01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
04-evaluation/lessons/02-ground-truth.md


In [73]:
##Vector
final_vec=embedder.encode(final_question)

In [74]:
vector_results=vecindex.search(final_vec,num_results=5)

for i in vector_results:
    print(i["filename"])

01-agentic-rag/lessons/01-intro.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/16-other-frameworks.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md


In [75]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [76]:
final_results=rrf([vector_results,text_results])

In [77]:
final_results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'